In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Qubit 측정하기

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하시길 권장합니다.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Qubit의 상태에 대한 정보를 얻으려면, [고전 비트(classical bit)](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Clbit)로 _측정_할 수 있습니다. Qiskit에서 측정은 계산 기저(computational basis), 즉 단일 Qubit의 Pauli-$Z$ 기저에서 수행됩니다. 따라서 측정은 Pauli-$Z$ 고유 상태 $|0\rangle$ 및 $|1\rangle$과의 겹침에 따라 0 또는 1을 산출합니다:

$$
|q\rangle \xrightarrow{measure}\begin{cases}
      0 (\text{outcome}+1), \text{with probability } p_0=|\langle q|0\rangle|^{2}\text{,} \\
      1 (\text{outcome}-1), \text{with probability } p_1=|\langle q|1\rangle|^{2}\text{.}
    \end{cases}
$$

## 회로 중간 측정
회로 중간 측정(Mid-circuit measurements)은 동적 Circuit의 핵심 구성 요소입니다. `qiskit-ibm-runtime` v0.43.0 이전에는 `measure`가 Qiskit에서 유일한 측정 명령어였습니다. 그러나 회로 중간 측정은 _종단 측정_(Terminal measurements, Circuit 끝에서 수행되는 측정)과 다른 튜닝 요구 사항을 갖습니다. 예를 들어, 회로 중간 측정을 튜닝할 때는 명령어 지속 시간을 고려해야 합니다. 더 긴 명령어일수록 Circuit에 더 많은 노이즈를 발생시키기 때문입니다. 종단 측정 이후에는 명령어가 없으므로 종단 측정의 경우 명령어 지속 시간을 고려할 필요가 없습니다.

`qiskit-ibm-runtime` v0.43.0에서 `MidCircuitMeasure` 명령어가 도입되었습니다. 이름에서 알 수 있듯이, 이는 IBM&reg; QPU에서의 회로 중간 측정에 최적화된 새로운 측정 명령어입니다.

> **Note:** `MidCircuitMeasure` 명령어는 Backend의 `supported_instructions`에 보고된 `measure_2` 명령어에 매핑됩니다. 그러나 `measure_2`는 모든 Backend에서 지원되지는 않습니다. `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)`를 사용하여 지원되는 Backend를 찾으세요. 향후에 새로운 측정 방식이 추가될 수 있지만, 이것이 보장되지는 않습니다.

## Circuit에 측정 적용하기
Circuit에 측정을 적용하는 방법에는 여러 가지가 있습니다:

### `QuantumCircuit.measure` 메서드
[`measure`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure) 메서드를 사용하여 [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class)을 측정합니다.

예시:

In [1]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(5, 5)
qc.x(0)
qc.x(1)
qc.x(4)
qc.measure(
    range(5), range(5)
)  #  Measures all qubits into the corresponding clbit.

In [2]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure(1, 0)  # Measure qubit 1 into the classical bit 0.

### `Measure` class

The Qiskit [Measure](/docs/api/qiskit/circuit#qiskit.circuit.Measure) class measures the specified qubits.

In [3]:
from qiskit.circuit import Measure

qc = QuantumCircuit(3, 1)
qc.x([0, 1])
qc.append(Measure(), [0], [0])  # measure qubit 0 into clbit 0

### `QuantumCircuit.measure_all` method

To measure all qubits into the corresponding classical bits, use the [`measure_all`](/docs/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) method. By default, this method adds new classical bits in a `ClassicalRegister` to store these measurements.

In [4]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_all()  # Measure all qubits.

### `Measure` 클래스
Qiskit의 [Measure](https://docs.quantum.ibm.com/api/qiskit/circuit#qiskit.circuit.Measure) 클래스는 지정된 Qubit을 측정합니다.

In [5]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_active()  # Measure qubits that are not idle, that is, qubits 0 and 2.

<span id="midcircuit"></span>
### `MidCircuitMeasure` method


Use `MidCircuitMeasure` to apply a mid-circuit measurement (requires `qiskit-ibm-runtime` v0.43.0 or later).  While you can use `QuantumCircuit.measure` for a mid-circuit measurement, because of its design, `MidCircuitMeasure` is typically a better choice.  For example, it adds less overhead to your circuit than when using `QuantumCircuit.measure`.

In [6]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.circuit import MidCircuitMeasure
from qiskit.circuit import Measure

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circ = QuantumCircuit(2, 2)
circ.x(0)
circ.append(MidCircuitMeasure(), [0], [0])
# circ.measure([0], [0])
# circ.measure_all()
print(circ.draw(cregbundle=False))

     ┌───┐┌────────────┐
q_0: ┤ X ├┤0           ├
     └───┘│            │
q_1: ─────┤  Measure_2 ├
          │            │
c_0: ═════╡0           ╞
          └────────────┘
c_1: ═══════════════════
                        


### `QuantumCircuit.measure_all` 메서드
모든 Qubit을 해당하는 고전 비트로 측정하려면 [`measure_all`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#measure_all) 메서드를 사용합니다. 기본적으로 이 메서드는 이러한 측정을 저장하기 위해 `ClassicalRegister`에 새로운 고전 비트를 추가합니다.